In [ ]:
import logging
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType, DateType

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

try:
    # Step 1: Load Data Sources
    logger.info("Loading data sources...")
    
    # Load manually entered data
    channel_df = spark.table("catalog.source_db.channel")
    manual_date_df = spark.table("catalog.source_db.manual_date")
    
    # Load database tables
    tdmedpod_df = spark.sql("SELECT * FROM catalog.source_db.tdmedpod")
    tableupdated_df = spark.sql("SELECT * FROM catalog.source_db.dbo_tableupdated_new_1000")

    # Step 2: Date Calculations
    logger.info("Performing date calculations...")
    
    current_date = datetime.now().strftime('%Y-%m-%d')
    manual_date_df = manual_date_df.withColumn("CurrentDate", F.lit(current_date))
    
    manual_date_df = manual_date_df.withColumn(
        "Prior_Week_Start",
        F.date_sub(F.date_trunc("week", F.col("CurrentDate")), 7)
    ).withColumn(
        "Prior_Week_End",
        F.date_sub(F.date_trunc("week", F.col("CurrentDate")), 1)
    ).withColumn(
        "Yesterday",
        F.date_sub(F.col("CurrentDate"), 1)
    ).withColumn(
        "Today",
        F.col("CurrentDate")
    ).withColumn(
        "P1M_Start",
        F.add_months(F.date_trunc("month", F.col("CurrentDate")), -1)
    ).withColumn(
        "P1M_End",
        F.date_sub(F.date_trunc("month", F.col("CurrentDate")), 1)
    )

    # Step 3: Data Aggregation
    logger.info("Aggregating data...")
    
    tdmedpod_agg_df = tdmedpod_df.groupBy("BILL_DTE").agg(
        F.sum("Invoices").alias("Sum_Invoices"),
        F.sum("LANDED_COST").alias("Sum_LANDED_COST"),
        F.sum("EXT_FINAL_PRICE").alias("Sum_EXT_FINAL_PRICE"),
        F.sum("Trans_Charge_Amt").alias("Sum_Trans_Charge_Amt"),
        F.sum("RESTOCK_Fee").alias("Sum_RESTOCK_Fee"),
        F.sum("Special_Hndl_Amt").alias("Sum_Special_Hndl_Amt"),
        F.sum("Vendor_Hndl_Amt").alias("Sum_Vendor_Hndl_Amt"),
        F.sum("MOC_Amt").alias("Sum_MOC_Amt"),
        F.sum("Fuel_Surcharge").alias("Sum_Fuel_Surcharge")
    )

    # Step 4: Custom Field Creation
    logger.info("Creating custom fields...")
    
    tdmedpod_agg_df = tdmedpod_agg_df.withColumn(
        "Rush_Order_Fee",
        F.col("Sum_Trans_Charge_Amt") + F.col("Sum_RESTOCK_Fee")
    ).withColumn(
        "BIA_SHIP_HNDL_AMT",
        F.col("Sum_Trans_Charge_Amt") + F.col("Sum_RESTOCK_Fee") + F.col("Sum_Special_Hndl_Amt") +
        F.col("Sum_Vendor_Hndl_Amt") + F.col("Sum_MOC_Amt") + F.col("Sum_Fuel_Surcharge")
    ).withColumn(
        "COE_SHIP_HNDL_AMT",
        F.col("BIA_SHIP_HNDL_AMT") + F.col("Rush_Order_Fee")
    )

    # Step 5: Data Cleansing
    logger.info("Performing data cleansing...")
    
    tdmedpod_agg_df = tdmedpod_agg_df.withColumn(
        "null_yn",
        F.when(F.col("FNC_ID").isNull() | (F.col("FNC_ID") == ""), "Y")
        .when(F.col("Whs").isNull() | (F.col("Whs") == ""), "Y")
        .otherwise("N")
    ).withColumn(
        "FNC_ID",
        F.when(F.col("FNC_ID").isNull() | (F.col("FNC_ID") == ""), "OTH").otherwise(F.col("FNC_ID"))
    ).withColumn(
        "FNC_DESC",
        F.when(F.col("FNC_DESC").isNull() | (F.col("FNC_DESC") == ""), "OTHER").otherwise(F.col("FNC_DESC"))
    )

    # Step 6: Union and Joins
    logger.info("Performing union and joins...")
    
    enriched_df = tdmedpod_agg_df.join(
        tableupdated_df,
        tdmedpod_agg_df["DIST_CHNL_ID"] == tableupdated_df["DIST_CHNL"],
        "inner"
    )

    # Step 7: Output Generation
    logger.info("Writing output to Unity Catalog...")
    
    spark.sql("DROP TABLE IF EXISTS catalog.target_db.final_output")
    enriched_df.write.format("delta").mode("overwrite").saveAsTable("catalog.target_db.final_output")

    logger.info("ETL workflow completed successfully!")

except Exception as e:
    logger.error(f"Error occurred during ETL workflow: {str(e)}")
    raise
